In [1]:
import pandas as pd
import numpy as np

In [2]:
pip install optuna

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [3]:
from sklearn.metrics import (
    mean_squared_error,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
import optuna
from functools import partial

In [7]:
class MLModels:
    def __init__(self, X, y, models, preprocessor=None, test_size=0.2, random_state=111):
        """
        Initialize the storm prediction model framework with Optuna integration.
        
        Parameters:
        - X: Features DataFrame
        - y: Target variable
        - models: Dictionary of models to evaluate (name: model instance)
        - preprocessor: sklearn preprocessor pipeline
        - test_size: Proportion for test split
        - random_state: Random seed for reproducibility
        """
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=test_size, random_state=random_state, stratify=y
        )
        
        self.X_train = X_train
        self.X_test = X_test
        self.y_train = y_train
        self.y_test = y_test
        self.random_state = random_state
        self.models = models
        self.preprocessor = preprocessor
        self.trained_models = {}
        self.results = pd.DataFrame()
        self.y_pred_proba = {}
        self.y_pred = {}
        self.studies = {}  
        self.best_params = {}  
        
    def set_models(self, models):
        """Update the models to be trained and evaluated."""
        self.models = models
        
    def _create_objective(self, model_name, pipeline):
        """Create an Optuna objective function for a specific model."""
        def objective(trial):
            params = {}
            
            if model_name == 'RandomForest':
                params = {
                    'n_estimators': trial.suggest_int('n_estimators', 50, 500),
                    'max_depth': trial.suggest_int('max_depth', 3, 30),
                    'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
                    'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
                    'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
                    'bootstrap': trial.suggest_categorical('bootstrap', [True, False])
                }
            elif model_name == 'GradientBoosting':
                params = {
                    'n_estimators': trial.suggest_int('n_estimators', 50, 500),
                    'learning_rate': trial.suggest_float('learning_rate', 0.001, 0.2, log=True),
                    'max_depth': trial.suggest_int('max_depth', 3, 10),
                    'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
                    'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
                    'subsample': trial.suggest_float('subsample', 0.5, 1.0),
                    'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None])
                }
            elif model_name == 'SVM':
                params = {
                    'C': trial.suggest_float('C', 0.1, 10, log=True),
                    'kernel': trial.suggest_categorical('kernel', ['linear', 'rbf', 'poly']),
                    'gamma': trial.suggest_categorical('gamma', ['scale', 'auto']),
                    'degree': trial.suggest_int('degree', 2, 4) if params.get('kernel') == 'poly' else 3
                }
            
            # Clone the pipeline and set parameters
            current_pipeline = pipeline.clone()
            current_pipeline.set_params(**{f'classifier__{k}': v for k, v in params.items()})
            
            # Use cross-validation to evaluate performance
            score = cross_val_score(
                current_pipeline,
                self.X_train,
                self.y_train,
                cv=5,
                scoring='roc_auc',
                n_jobs=-1
            ).mean()
            
            return score
        
        return objective
    
    def train_models(self, n_trials=50, timeout=None, direction='maximize', n_jobs=-1):
        """
        Train all specified models with Optuna hyperparameter optimization.
        
        Parameters:
        - n_trials: Number of optimization trials per model
        - timeout: Maximum time in seconds to spend on optimization per model
        - direction: Optimization direction ('maximize' or 'minimize')
        """
        self.trained_models = {}
        self.best_params = {}
        self.studies = {}
        
        for name, model in self.models.items():
            print(f"\nOptimizing {name} with Optuna...")
            
            # Create pipeline with preprocessor if it exists
            if self.preprocessor is not None:
                pipeline = Pipeline([
                    ('preprocessor', self.preprocessor),
                    ('classifier', model)
                ])
            else:
                pipeline = Pipeline([('classifier', model)])
            
            # Create study
            study = optuna.create_study(
                direction=direction,
                sampler=optuna.samplers.TPESampler(seed=self.random_state)
            )
            
            # Optimize
            objective = self._create_objective(name, pipeline)
            study.optimize(objective, n_trials=n_trials, timeout=timeout, n_jobs=n_jobs)
            
            self.studies[name] = study
            self.best_params[name] = study.best_params
            
            # Train final model with best parameters
            pipeline.set_params(**{f'classifier__{k}': v for k, v in study.best_params.items()})
            pipeline.fit(self.X_train, self.y_train)
            self.trained_models[name] = pipeline
            
            print(f"Best parameters for {name}: {study.best_params}")
            print(f"Best ROC AUC: {study.best_value:.4f}")
    
    def evaluate_models(self):
        """Evaluate all trained models on the test set."""
        metrics = []
        
        for name, model in self.trained_models.items():
            # Get predictions
            y_pred = model.predict(self.X_test)
            y_pred_proba = model.predict_proba(self.X_test)[:, 1]  # Probability of positive class
            
            self.y_pred[name] = y_pred
            self.y_pred_proba[name] = y_pred_proba
            
            model_metrics = {
                'Model': name,
                'Accuracy': accuracy_score(self.y_test, y_pred),
                'Precision': precision_score(self.y_test, y_pred),
                'Recall': recall_score(self.y_test, y_pred),
                'F1': f1_score(self.y_test, y_pred),
                'ROC AUC': roc_auc_score(self.y_test, y_pred_proba),
                'Confusion Matrix': confusion_matrix(self.y_test, y_pred),
                'Classification Report': classification_report(self.y_test, y_pred, output_dict=True)
            }
            
            metrics.append(model_metrics)
        
        self.results = pd.DataFrame(metrics).set_index('Model')
    
    def get_feature_importances(self):
        """Get feature importances for tree-based models."""
        importances = {}
        
        for name, model in self.trained_models.items():
            classifier = model.named_steps['classifier']
            
            if hasattr(classifier, 'feature_importances_'):
                # Get feature names after preprocessing
                if self.preprocessor is not None:
                    try:
                        feature_names = self.preprocessor.get_feature_names_out()
                    except AttributeError:
                        feature_names = self.X_train.columns.tolist()
                else:
                    feature_names = self.X_train.columns.tolist()
                    
                importances[name] = pd.Series(
                    classifier.feature_importances_,
                    index=feature_names
                ).sort_values(ascending=False)
                
        return importances
    
    def get_optuna_visualizations(self, model_name):
        """
        Generate Optuna visualizations for a specific model.
        
        Returns:
        - optimization_history: Plot of optimization history
        - param_importances: Plot of parameter importances
        """
        if model_name not in self.studies:
            raise ValueError(f"No study found for model: {model_name}")
            
        study = self.studies[model_name]
        
        history = optuna.visualization.plot_optimization_history(study)
        importances = optuna.visualization.plot_param_importances(study)
        
        return history, importances
    
    def __str__(self):
        return f"StormPredictionModels with {len(self.models)} models: {list(self.models.keys())}"